# Answers: 7 · Scaling up

Worked answers to the **Try it yourself** exercises at the end of
[`7_scaling_up.ipynb`](../7_scaling_up.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

In [ ]:
import sys; sys.path.insert(0, "..")
import time, math
import numpy as np
from tour import *
smited = np.load("/N/project/SingleCell_Image/mengjing/enhits_large/embed/smited_embeddings.npy", mmap_mode="r")
big = np.load("/N/project/SingleCell_Image/mengjing/enhits_large/embed/molformer_embeddings.npy", mmap_mode="r")
print("ready")

---
## Exercise 1: only 50 rows. Is scattered reading still slower?

New random rows every run, so the computer can't reuse rows it read before.

In [ ]:
for n in [50, 500]:
    start = np.random.default_rng().integers(0, len(smited) - n)
    t = time.time(); np.array(smited[start:start + n]); together = time.time() - t
    rows = np.sort(np.random.default_rng().choice(len(smited), n, replace=False))
    t = time.time(); smited[rows]; scattered = time.time() - t
    print(f"{n:4d} rows: together {together:.3f} s, scattered {scattered:.3f} s  "
          f"-> {scattered / max(together, 1e-4):,.0f}x slower, {1000 * scattered / n:.1f} ms per scattered row")

**Answer.** Yes, still much slower: about 20× for 50 rows here, against about 40× for 500 rows in the same run. Look at the cost per row: each scattered row costs about 20–27 milliseconds, whether you read 50 or 500. Every jump to a new place on the network disk pays a fixed price, so reading fewer scattered rows doesn't make each one cheaper. (Your numbers will differ: they depend on how busy the shared storage is.)

---
## Exercise 2: how long would the whole 2.1M-row file take in chunks?

Time a few chunks from a part of the file the lesson didn't read, then scale up.

In [ ]:
chunk, n_chunks = 50_000, 4
first = 1_200_000                                   # the lesson read rows 0-500,000; start elsewhere
t = time.time()
for start in range(first, first + chunk * n_chunks, chunk):
    np.asarray(big[start:start + chunk]).sum(dtype=np.float64)
seconds = time.time() - t
rows_per_s = chunk * n_chunks / seconds
print(f"{chunk * n_chunks:,} rows in {seconds:.1f} s  ->  {rows_per_s:,.0f} rows/s")
print(f"whole file ({len(big):,} rows): about {len(big) / rows_per_s:.0f} s")

**Answer.** At about 6,500 rows per second, the whole 2,104,318-row file takes about 5 minutes (≈ 320 s) to read in chunks. That's for **one** backbone, read **once**. The project does this every round, for several backbones, which is why `ParallelMVEExplorer` spreads the work over many GPUs and the prefilter (notebook 6) avoids re-scoring the whole pool. Your speed will vary with how busy the storage is.

---
## Exercise 3: shards for D4 so each is under 5 million molecules

In [ ]:
from smiles_chunking import _chunk_bounds
n_d4, limit = 116_241_184, 5_000_000
k = math.ceil(n_d4 / limit)                         # the smallest whole number of shards
sizes = [e - s for s, e in (_chunk_bounds(n_d4, i, k) for i in range(k))]
print(f"{k} shards; largest shard {max(sizes):,} molecules")
sizes_one_fewer = [e - s for s, e in (_chunk_bounds(n_d4, i, k - 1) for i in range(k - 1))]
print(f"with {k - 1} shards the largest would be {max(sizes_one_fewer):,}  (over the limit)")

**Answer.** 24 shards. 116,241,184 ÷ 5,000,000 = 23.2, and you can't have part of a shard, so round **up** (`math.ceil`). With 24 the largest shard holds 4,843,383 molecules. With 23 it would hold 5,053,965, just over the limit. Using the project's real `_chunk_bounds` to check matters, because it decides exactly where each shard starts and ends.